# 17 — Train one member of the two-model diversity experiment

Run this notebook twice on an **A100/H100 80GB**, first with `MODEL_INDEX=0`, then with `1`.
Both runs use one shared manifest: every LIBERO task is retained, while demonstrations are
independently sampled with replacement within task. This is a full-model fine-tune by default.

The start point is the agreed raw `lerobot/pi05_base`. Neither member inherits the common
LIBERO-finetuned solution; their LIBERO specialization comes only from their independently
bootstrapped demonstration multisets. Final weights are pushed to your Hugging Face account.
The shared split manifest is stored in Drive so both runs use exactly the same experiment.

This is a **diversity-signal pilot**, not a reproduction of published LIBERO performance. The
official OpenPI recipe trains raw pi0.5 for 30k steps with batch size 256 and a 10-action horizon;
this pilot uses 3k steps, batch size 16, and keeps the 50-action chunk / execute-10 interface used
by the PnP experiments. The training shim rebuilds the raw checkpoint's stale processor metadata
with the exact pinned LeRobot implementation; it does not alter the model weights or turn on an
extra relative-action transform.

## 1. Install the exact pinned training stack

In [ ]:
EXTRAS = 'train'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Configuration

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi
from google.colab import drive

drive.mount("/content/drive")

MODEL_INDEX = 0                 # rerun a separate copy with 1
STEPS = 3000                    # signal pilot; increase only after both models train cleanly
BATCH_SIZE = 16                 # safe starting point on an 80GB A100/H100
SAVE_FREQ = 10000               # greater than STEPS: save only the final checkpoint
SAVE_CHECKPOINTS = False        # False: no local/Drive checkpoints; final model still uploads to HF
FULL_FINETUNE = True            # planned experiment; False is explicit expert-only fallback
COMPILE_MODEL = False           # avoids a long first-step compile during the pilot
WANDB = False
RESUME = False

HF_USER = HfApi().whoami()["name"]
MODEL_REPOS = [f"{HF_USER}/pi05-base-to-libero-bootstrap-m0-v1",
               f"{HF_USER}/pi05-base-to-libero-bootstrap-m1-v1"]
PERSISTENT_ROOT = Path("/content/drive/MyDrive/pnp_diversity")
MANIFEST_PATH = PERSISTENT_ROOT / "bootstrap_manifest.json"
OUTPUT_DIR = Path(f"/content/pi05_diversity/train_m{MODEL_INDEX}")
CHECKPOINT_MIRROR_DIR = PERSISTENT_ROOT / f"checkpoint_m{MODEL_INDEX}"
print({"member": MODEL_INDEX, "model_repo": MODEL_REPOS[MODEL_INDEX],
       "output": str(OUTPUT_DIR), "checkpoint_mirror": str(CHECKPOINT_MIRROR_DIR),
       "full_finetune": FULL_FINETUNE, "save_checkpoints": SAVE_CHECKPOINTS})

## 3. Build or load the shared episode-bootstrap manifest

The file in Drive contains both independently sampled members. Member 1 must load this exact file;
it must not generate a second manifest.

In [ ]:
from pnp.diversity import (bootstrap_manifest_summary,
    build_bootstrap_manifest_from_lerobot, load_bootstrap_manifest,
    save_bootstrap_manifest)

if MANIFEST_PATH.exists():
    manifest = load_bootstrap_manifest(MANIFEST_PATH)
else:
    manifest = build_bootstrap_manifest_from_lerobot()
    save_bootstrap_manifest(manifest, MANIFEST_PATH)
assert manifest["source_model"] == "lerobot/pi05_base", manifest["source_model"]
display(bootstrap_manifest_summary(manifest))
print("manifest:", MANIFEST_PATH)
print("manifest hash:", manifest["manifest_hash"])
print("dataset revision:", manifest["dataset_revision"])
print("raw model revision:", manifest["source_model_revision"])
print("tasks:", manifest["n_tasks"], "source episodes:", manifest["n_source_episodes"])

## 4. Launch training

Expected after the weights load: `Built fresh pinned-LeRobot processors with LIBERO camera
mapping.` This confirms that the obsolete processor metadata in `pi05_base` was replaced by the
pinned implementation before optimizer creation.

Training stays on fast local disk. After each complete save, the checkpoint is verified and
mirrored to Drive, then all local checkpoint copies and older Drive mirrors are removed. If a
fresh Colab runtime starts with `RESUME=True`, the newest complete Drive checkpoint is preferred;
`/content` is used only if Drive has no complete checkpoint. Partial saves are ignored. The
temporary checkpoint copy is removed after model, optimizer, and scheduler state are loaded.
Do not manually delete checkpoint folders while a save is in progress.

With the default `SAVE_CHECKPOINTS=False`, LeRobot writes no training checkpoints at all; the final
model and processors are still pushed to `MODEL_REPOS[MODEL_INDEX]` after step 3000. The tradeoff is
that a disconnect during this run cannot be resumed from a newer step.

In [ ]:
import subprocess, sys
from pathlib import Path

args = [sys.executable, "-u",
        str(Path(package_dir) / "scripts" / "train_pi05_bootstrap.py"),
        "--manifest", str(MANIFEST_PATH), "--member", str(MODEL_INDEX),
        "--output-dir", str(OUTPUT_DIR),
        "--checkpoint-mirror-dir", str(CHECKPOINT_MIRROR_DIR),
        "--policy-repo-id", MODEL_REPOS[MODEL_INDEX],
        "--steps", str(STEPS), "--batch-size", str(BATCH_SIZE),
        "--save-freq", str(SAVE_FREQ)]
if not FULL_FINETUNE: args.append("--expert-only")
if COMPILE_MODEL: args.append("--compile-model")
if WANDB: args.append("--wandb")
if RESUME: args.append("--resume")
if not SAVE_CHECKPOINTS: args.append("--no-checkpoints")
print("starting member", MODEL_INDEX)
process = subprocess.Popen(
    args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1)
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
if return_code:
    raise RuntimeError(f"Training exited with code {return_code}")

## 5. Record the immutable identifiers

In [ ]:
print({"member": MODEL_INDEX, "model_repo": MODEL_REPOS[MODEL_INDEX],
       "manifest_hash": manifest["manifest_hash"], "source_model": manifest["source_model"],
       "steps": STEPS, "batch_size": BATCH_SIZE, "full_finetune": FULL_FINETUNE})
print("Before training member 1, preserve and reuse:", MANIFEST_PATH)